<a href="https://colab.research.google.com/github/IsabellaHannaCSM/skills-introduction-to-github/blob/main/calculo_tempo_compra_por_curso.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Análise de Tempo até Compra por Curso
Este notebook calcula o número de dias que um lead leva para se tornar cliente no HubSpot, separando por tipo de pipeline (Lato Sensu ou Stricto Sensu) e agrupando por curso.

In [1]:
!pip install pandas openpyxl

In [6]:
from google.colab import files

uploaded = files.upload()


Saving negocios.xlsx to negocios.xlsx


In [7]:
import pandas as pd

# Caminho para seu arquivo Excel
arquivo = 'negocios.xlsx'

# Carregando apenas as colunas necessárias da aba 'Base de Negócios'
df = pd.read_excel(
    arquivo,
    sheet_name="Base de Negócios",
    usecols=[
        "Curso de Interesse",
        "Pipeline",
        "Data de criação",
        "Etapa do negócio",
        "Data de fechamento",
        'Date entered "Negócio Fechado (Lato Sensu)"'
    ]
)

# Visualizar as primeiras linhas
df.head()

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Pipeline,Data de criação,Etapa do negócio,Curso de Interesse,Data de fechamento,"Date entered ""Negócio Fechado (Lato Sensu)"""
0,Lato Sensu,2025-11-06 09:59:01.949,Entrada (Lato Sensu),Direito Constitucional,NaT,NaT
1,Stricto Sensu,2025-11-06 09:25:52.940,Entrada MQL (Stricto Sensu),"Mestrado Profissional em Direito, Justiça e De...",NaT,NaT
2,Lato Sensu,2025-11-06 09:25:09.624,Entrada (Lato Sensu),Licitações e Contratos,NaT,NaT
3,Lato Sensu,2025-11-06 09:23:26.190,Conexão (Lato Sensu),MBA em Gestão Pública e Políticas Públicas,NaT,NaT
4,Lato Sensu,2025-11-06 09:11:58.830,Conexão (Lato Sensu),LLM em Direito Penal Econômico,NaT,NaT


In [8]:
# Converter colunas de datas
df["Data de criação"] = pd.to_datetime(df["Data de criação"], errors="coerce")
df["Data de fechamento"] = pd.to_datetime(df["Data de fechamento"], errors="coerce")
df['Date entered "Negócio Fechado (Lato Sensu)"'] = pd.to_datetime(
    df['Date entered "Negócio Fechado (Lato Sensu)"'], errors="coerce"
)

# Filtrar apenas negócios fechados
df = df[df["Etapa do negócio"].str.contains("Negócio Fechado", na=False)].copy()
print(f"Negócios fechados encontrados: {len(df)}")

Negócios fechados encontrados: 6333


In [9]:
# Função para calcular dias até a compra
def calcular_dias(row):
    if row["Pipeline"] == "Lato Sensu":
        return (row['Date entered "Negócio Fechado (Lato Sensu)"'] - row["Data de criação"]).days
    elif row["Pipeline"] == "Stricto Sensu":
        return (row["Data de fechamento"] - row["Data de criação"]).days
    else:
        return None

df["Dias até compra"] = df.apply(calcular_dias, axis=1)
df.head()

,Pipeline,Data de criação,Etapa do negócio,Curso de Interesse,Data de fechamento,"Date entered ""Negócio Fechado (Lato Sensu)""",Dias até compra
182,Lato Sensu,2025-11-04 16:35:19.185,Negócio Fechado (Lato Sensu),Advocacia em Direito Privado e Empresarial,2025-11-05 11:00:33.989,2025-11-05 11:00:33.989,0.0
849,Lato Sensu,2025-10-30 13:33:49.194,Negócio Fechado (Lato Sensu),MBA em Relações Institucionais e Governamentais,2025-11-05 18:39:25.205,2025-11-05 18:39:25.205,6.0
1074,Lato Sensu,2025-10-30 10:14:44.053,Negócio Fechado (Lato Sensu),Direito Constitucional,2025-11-05 12:21:51.812,2025-11-05 12:21:51.812,6.0
1434,Lato Sensu,2025-10-27 13:30:15.097,Negócio Fechado (Lato Sensu),Direito Constitucional,2025-11-05 17:43:26.551,2025-11-05 17:43:26.551,9.0
1965,Lato Sensu,2025-10-16 14:55:14.098,Negócio Fechado (Lato Sensu),MBA em Segurança Pública,2025-10-16 15:41:32.434,2025-10-16 15:41:32.434,0.0


In [10]:
# Sheet 1: dados linha a linha
df_detalhado = df[[
    "Curso de Interesse", "Pipeline", "Data de criação",
    "Data de fechamento", 'Date entered "Negócio Fechado (Lato Sensu)"',
    "Dias até compra"
]]

# Sheet 2: agregação por curso
df_resumo = df_detalhado.groupby("Curso de Interesse").agg(
    Total_de_Vendas=("Dias até compra", "count"),
    Media_dias=("Dias até compra", "mean"),
    Min_dias=("Dias até compra", "min"),
    Max_dias=("Dias até compra", "max")
).reset_index()

df_resumo.head()

,Curso de Interesse,Total_de_Vendas,Media_dias,Min_dias,Max_dias
0,Advocacia em Direito Privado e Empresarial,50,11.600000,0.0,107.0
1,Direito Administrativo,80,9.437500,0.0,73.0
2,Direito Administrativo; LLM em Direito dos Neg...,0,NaN,NaN,NaN
3,Direito Constitucional,59,8.423729,0.0,110.0
4,Direito Constitucional; Direito Eleitoral,0,NaN,NaN,NaN


In [11]:
# Exportar para Excel
with pd.ExcelWriter("tempo_ate_compra_por_curso_COMPLETO.xlsx", engine="openpyxl") as writer:
    df_resumo.to_excel(writer, sheet_name="Resumo por Curso", index=False)
    df_detalhado.to_excel(writer, sheet_name="Detalhado", index=False)

print("Arquivo Excel gerado com sucesso!")

Arquivo Excel gerado com sucesso!
